# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *23* |
| **Integrantes** | *Manuela Tangarife Zapata* |
| **Caso de estudio** | *Wanderbricks* |
| **Fecha de entrega** | domingo 6 de septiembre |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

---
## 1. Contexto y problema

Wanderbircks es un marketplace de alquiler vacacional que maneja información sobre usuarios, anfitriones, propiedades, reservas, pagos, reseñas y comportamiento de navegación. Para analizar esta información no basta solo con tener los datos. También se necesita un entorno que permita almacenarlos, procesarlos y administrarlos de forma organizada. 

El entorno debe dejar trabajar con distintos tipos de datos. Debe mantener una separación entre los datos originales y los datos transformados. Además tiene que facilitar el acceso según las responsabilidades de cada usuario. También debe permitir que los procesos se automaticen para evitar que las actualizaciones dependan solo de una ejecución manual. 

Por esta razón, propongo un entorno en Databricks organizado en capas Bronze, Silver y Gold. En este entorno, los datos pasan desde su estado original hasta convertirse en información lista para el análisis. Tambien se tienen en cuenta los permisos, el gobierno, el linaje y la automatización, para que la solución se mantenga y crezca con el tiempo.

---
## 2. Descripción de los datos

En la actividad anterior se trabajó con nueve tablas del caso Wanderbricks: bookings, payments, users, properties, hosts, destinations, reviews, clickstream y customer_support_logs. Estas tablas en conjunto contienen 545.676 registros y representan información de reservas, pagos, usuarios, propiedades, destinos, reseñas, navegación y soporte. 

Los datos son principalmente estructurados, aunque también hay elementos semiestructurados, como los campos anidados de clickstream y los mensajes almacenados en customer_support_logs. Algunas tablas también tienen texto libre, por ejemplo las reseñas y los registros de soporte.

En esta actividad planteo una actualización periódica del entorno, mediante una programación diaria para los procesos automatizados. Los principales usuarios serían los analistas, los ingenieros de datos y los administradores del entorno.

---
## 3. Decisiones de diseño y justificación
### 3.1 Diagrama de la arquitectura

![Diagrama EA2.drawio_1789931265390.png](./Diagrama EA2.drawio_1789931265390.png "Diagrama EA2.drawio_1789931265390.png")

En esta arquitectura, Databricks gestiona la infraestructura y los componentes adminitrados de la plataforma, y el equipo define la organización lógica de los datos, las transformaciones, los permisos, procesos y automatizaciones.

En la capa Bronze se guardan los datos originales, en Silver se hacen las transformaciones y valididaciones necesarias y en Gold se generan datos listos para el análisis. Los datos pueden ser consultados mediante SQL y PySpark, y con Catalog Explorer se pueden administrar y consultar elementos del entorno.

### 3.2 Matriz de roles

Los permisos se plantean en función de cada rol. El analista debe poder consultar la información para hacer sus análisis, pero no debe poder modificar los datos originales. El ingeniero de datos tiene que trabajar con los procesos de transformación y mantenimiento de las tablas. El administrador es responsable de la seguridad, los permisos y el gobierno del entorno.

| Rol | Bronce | Plata | Oro |
|---|---|---|---|
| Analista | Lectura imitada | Lectura | Lectura |
| Ingeniero de datos | Lectura y escritura | Lectura y escritura | Creación y mantenimiento |
| Administrador | Gestión completa | Gestión completa | Gestión Completa |

### 3.3 Especificación del equivalente IaaS

Como equivalente de laaS se plantea la ejecución de la solución en máquinas virtuales que administradas.

La idea es una máquina para la orquestación de procesos utilizando Airflow, una máquina principal para Spark y dos máquinas worker para distribuir el procesamiento. Se trabajaría sobre el sistema operativo Ubuntu Server LTS. 

La máquina principal podría tener alrededor de 4 vCPU y 16 GB de memoria, mientras que cada worker tendría cerca de 8 vCPU y 32 GB de memoria. Se contempla un espacio de almacenamiento inicial de 500 GB, con posibilidad de ampliación en función del crecimiento de los datos. 

La infraestructura estaría dentro de una red privada, con acceso administrativo restringido y reglas de seguridad para separar el tráfico interno del acceso externo. En las máquinas habría que instalar y configurar Spark, Python, Airflow y las dependencias necesarias para ejecutar los procesos.

### 3.4 Comparación IaaS / PaaS / SaaS

| Criterio | IaaS | PaaS | SaaS |
|---|---|---|---|
| Control | Alto | Medio | Bajo |
| Tiempo hasta el primer resultado | Mayor | Menor | Muy bajo |
| Esfuerzo operativo | Alto | Medio o bajo | Bajo |
| Costo | Depende de la infrestructura y operación | Depende del uso de la plataforma | Normalmente por suscripción o uso |
| Escalabilidad | Requiere configuración | Facilitada por la plataforma | Principalmente administrada por el proveedor |
| Gobierno |Mayor control, pero mayor responsabilidad | Control centralizado con servicios administrados | Depende de las opciones del producto|

**Conclusión:** Para el caso de Wanderbricks considero conveniente utilizar una plaataforma administrada como Databricks, ya que permite trabajar con los datos y los procesos sin tener que administrar directamente las máquinas, sistema operativo y gran parte de la infraestructura. Esto permite concentración en la organización de los datos, las transformaciones, los permisos y los procesos analíticos.

El enfoque laaS tendría sentido si se necesitara un mayor control sobre la infraestructura y la configuración de las máquinas. Por otro lado un modelo SaaS estaría más orientado al consumo de una aplicación ya terminada y ofrecería menos control sobre la infraestructura y los procesos internos.

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# Revice la organización actual del entorno

CATALOGO = "bigdata_grupo23"
ESQUEMA = "wanderbricks"

print("Catalogo utilizado:", CATALOGO)
print("Esquema utilizado:", ESQUEMA)

print("Esquemas disponibles:")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

print("Volúmenes disponibles:")
display(spark.sql(f"SHOW VOLUMES IN {CATALOGO}.{ESQUEMA}"))

print("Tablas del esquema:")
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}"))

### 4.2 Permisos

*Al menos dos sentencias GRANT con niveles distintos sobre objetos distintos.*

In [0]:
# Primero revisamos los grupos disponibles en Databricks.
# Necesitamos sus nombres exactos para poder hacer los GRANT.

display(spark.sql("SHOW GROUPS"))

In [0]:
# Primero vamos a saber cuál es el usuario que esta conectado a Databricks.
# Usaremos este usuario para hacer los permisos.

USUARIO = spark.sql(
    "SELECT session_user() AS usuario"
).first()["usuario"]

print("Usuario actual:", USUARIO)

In [0]:
# GRANT 1 — a quién, qué y por qué:

spark.sql("""
GRANT SELECT
ON TABLE bigdata_grupo23.wanderbricks.destinations_gold
TO `manuela.tangarife@est.iudigital.edu.co`	
""")

print("GRANT 1 ejecutado correctamente")

# GRANT 2 — a quién, qué y por qué:

spark.sql("""
GRANT MODIFY 
ON TABLE bigdata_grupo23.wanderbricks.bookings_bronze
TO `manuela.tangarife@est.iudigital.edu.co`
""")

print("GRANT 2 ejecutado correctamente.")

# display(spark.sql(f"SHOW GRANTS ON TABLE {TABLA}"))

print("Permisos de destinations_gold:")
display(
    spark.sql("""
    SHOW GRANTS ON TABLE bigdata_grupo23.wanderbricks.destinations_gold
    """)
)

print("Permisos de bookings_bronze:")
display(
    spark.sql("""
    SHOW GRANTS ON TABLE bigdata_grupo23.wanderbricks.bookings_bronze
    """)
)

### 4.3 Linaje

![image_1789940981439.png](./image_1789940981439.png "image_1789940981439.png")

In [0]:
spark.sql("""
DESCRIBE TABLE bigdata_grupo23.wanderbricks.destinations_gold
""").show(truncate=False)

destinations_gold.printSchema()

In [0]:
spark.sql("""
DROP TABLE IF EXISTS bigdata_grupo23.wanderbricks.destinations_gold
""")

print("Tabla destinations_gold anterior eliminada correctamente.")

In [0]:
from pyspark.sql import functions as F

bookings = spark.table("bigdata_grupo23.wanderbricks.bookings_silver")
properties = spark.table("bigdata_grupo23.wanderbricks.properties_silver")
destinations = spark.table("bigdata_grupo23.wanderbricks.destinations_silver")

destinations_gold = (
    bookings
    .filter(F.col("status") != "revision")
    .join(
        properties,
        bookings.property_id == properties.property_id,
        "inner"
    )
    .join(
        destinations,
        properties.destination_id == destinations.destination_id,
        "inner"
    )
    .groupBy(
        destinations.destination,
        destinations.country
    )
    .agg(
        F.count(bookings.booking_id).alias("total_reservas"),
        F.round(F.sum(bookings.total_amount), 2).alias("valor_total_reservas"),
        F.round(F.avg(bookings.total_amount), 2).alias("valor_promedio_reserva")
    )
    .withColumn("actualizado_en", F.current_timestamp())
)

destinations_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo23.wanderbricks.destinations_gold"
    )

print("Tabla Gold creada correctamente.")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM bigdata_grupo23.wanderbricks.destinations_gold
    ORDER BY valor_total_reservas DESC
    """)
)

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida.
Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*

In [0]:
from pyspark.sql import functions as F

bookings = spark.table("bigdata_grupo23.wanderbricks.bookings_silver")
properties = spark.table("bigdata_grupo23.wanderbricks.properties_silver")
destinations = spark.table("bigdata_grupo23.wanderbricks.destinations_silver")

# Se excluye revición porque fue usado como prueba Delta en la EA1.
destinations_gold = (
    bookings
    .filter(F.col("status") != "revision")
    .join(
        properties,
        bookings.property_id == properties.property_id,
        "inner"
    )
    .join(
        destinations,
        properties.destination_id == destinations.destination_id,
        "inner"
    )
    .groupBy(
        destinations.destination,
        destinations.country
    )
    .agg(
        F.count(bookings.booking_id).alias("total_reservas"),
        F.round(F.sum(bookings.total_amount), 2).alias("valor_total_reservas"),
        F.round(F.avg(bookings.total_amount), 2).alias("valor_promedio_reserva")
    )
        .withColumn("actualizado.en", F.current_timestamp())
)

destinations_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo23.wanderbricks.destinations_gold"
)

print("Gold actualizado correctamente.")

In [0]:
from pyspark.sql import functions as F

gold = spark.table(
    "bigdata_grupo23.wanderbricks.destinations_gold"
)

total_filas = gold.count()

duplicados = (
    gold
    .groupBy("destination", "country")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

nulos = gold.filter(
    F.col("destination").isNull() |
    F.col("country").isNull()
).count()

print("Total de filas:", total_filas)
print("Total de duplicados:", duplicados)
print("Filas con datos nulos:", nulos)

if total_filas == 0:
    raise ValueError("La tabla Gold no tiene registros.")
if duplicados > 0:
    raise ValueError("Se encontraron destinos duplicados.")
if nulos > 0:
    raise ValueError("Se encontraron valores nulos.")

print("Validación completada correctamente")

---
## 5. Resultados

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Manuela Tangarife Zapata | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.
3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?

---
## ✅ Antes de entregar

- [ ] El diagrama de arquitectura está incluido y descrito
- [ ] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [ ] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [ ] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todo confirmado en /ea2 y el HTML subido a Canvas